CN7023-SEMCD-14_UdayKiranMudu





## 1. Environment Setup & Spark Initialization

In [ ]:
!pip install pyspark -q

import os
import tarfile
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, lower, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
# Initialize Spark Session with memory allocated for NLP processing
spark = SparkSession.builder \
    .appName("IMDB_Sentiment_Linear_SVM") \
    .config("spark.driver.memory", "4g") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Successfully Created!")

Spark Session Successfully Created!


## 2. Dataset Downloading & Parsing

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
tar_path = "aclImdb_v1.tar.gz"

if not os.path.exists(tar_path):
    print("Downloading IMDB Dataset...")
    r = requests.get(url, stream=True)
    with open(tar_path, "wb") as f:
        f.write(r.content)
    print("Download Complete. Extracting archive...")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall()
    print("Extraction Complete.")

Download Complete. Extracting archive...


/tmp/ipykernel_5029/1604121796.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Extraction Complete.


In [ ]:
def parse_imdb_dir(base_dir):
    """Parses text files into tuples of (review_text, label)."""
    data = []
    for sentiment in ['pos', 'neg']:
        dir_path = os.path.join(base_dir, sentiment)
        label = 1.0 if sentiment == 'pos' else 0.0
        for file_name in os.listdir(dir_path):
            if file_name.endswith('.txt'):
                with open(os.path.join(dir_path, file_name), 'r', encoding='utf-8') as f:
                    data.append((f.read(), label))
    return data

# Parse raw train and test splits
train_data = parse_imdb_dir("aclImdb/train")
test_data = parse_imdb_dir("aclImdb/test")

# Convert to PySpark DataFrames
train_df = spark.createDataFrame(train_data, ["review", "label"])
test_df = spark.createDataFrame(test_data, ["review", "label"])

print(f"Training Set Count: {train_df.count()}")
print(f"Testing Set Count: {test_df.count()}")

Training Set Count: 25000
Testing Set Count: 25000


## 3. Text Preprocessing & Data Exploration

In [ ]:
# Clean HTML tags (<br />) and non-alphabetical characters, convert to lowercase
def clean_text_df(df):
    return df.withColumn("cleaned_review", regexp_replace(col("review"), "<br\\s*/?>", " ")) \
             .withColumn("cleaned_review", regexp_replace(col("cleaned_review"), "[^a-zA-Z\\s]", "")) \
             .withColumn("cleaned_review", lower(col("cleaned_review")))

train_cleaned = clean_text_df(train_df)
test_cleaned = clean_text_df(test_df)

print("\nDataset Class Distribution (Training):")
train_cleaned.groupBy("label").count().show()

print("Sample Cleaned Review:")
train_cleaned.select("cleaned_review", "label").show(2, truncate=100)


Dataset Class Distribution (Training):
+-----+-----+
|label|count|
+-----+-----+
|  1.0|12500|
|  0.0|12500|
+-----+-----+

Sample Cleaned Review:
+----------------------------------------------------------------------------------------------------+-----+
|                                                                                      cleaned_review|label|
+----------------------------------------------------------------------------------------------------+-----+
|i dont know how anyone could hate this movie it is so funny it took a unique mind to come up with...|  1.0|
|this is a great film the first time i saw it i thought it was absorbing from start to finish and ...|  1.0|
+----------------------------------------------------------------------------------------------------+-----+
only showing top 2 rows


## 4. Spark ML Feature Pipeline (TF-IDF & Text Transformation)

In [ ]:
# 1. Tokenization: Split text into words
tokenizer = Tokenizer(inputCol="cleaned_review", outputCol="words")

# 2. Stop-words Removal: Filter common words (e.g., 'the', 'is')
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

# 3. Term Frequency (TF): Convert word tokens to feature vectors
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=20000)

# 4. Inverse Document Frequency (IDF): Rescale feature vectors based on document frequency
idf = IDF(inputCol="rawFeatures", outputCol="features")

# Build Pipeline
nlp_pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf])

In [ ]:
# Fit and Transform Training & Testing sets
pipeline_model = nlp_pipeline.fit(train_cleaned)
train_features = pipeline_model.transform(train_cleaned).select("features", "label").cache()
test_features = pipeline_model.transform(test_cleaned).select("features", "label").cache()

print("TF-IDF Feature Extraction Completed!")

TF-IDF Feature Extraction Completed!


## 5. Linear SVM Model Training

In [ ]:
# Linear Support Vector Machine Classifier
lsvc = LinearSVC(featuresCol="features", labelCol="label", maxIter=20, regParam=0.1)

print("\nTraining Linear SVM Model...")
lsvc_model = lsvc.fit(train_features)
print("Model Training Complete!")


Training Linear SVM Model...
Model Training Complete!


## 6. Inference & Prediction Generation

In [ ]:
predictions = lsvc_model.transform(test_features)

print("\nSample Predictions on Test Dataset:")
predictions.select("label", "prediction", "rawPrediction").show(10)


Sample Predictions on Test Dataset:
+-----+----------+--------------------+
|label|prediction|       rawPrediction|
+-----+----------+--------------------+
|  1.0|       1.0|[-0.4936458309114...|
|  1.0|       1.0|[-2.2014236755806...|
|  1.0|       1.0|[-0.5259788519237...|
|  1.0|       0.0|[0.93130321951769...|
|  1.0|       1.0|[-1.9164862592222...|
|  1.0|       1.0|[-0.7440627182701...|
|  1.0|       1.0|[-1.5502185930602...|
|  1.0|       1.0|[-0.4670518321787...|
|  1.0|       1.0|[-0.6472071630668...|
|  1.0|       0.0|[0.04633345346176...|
+-----+----------+--------------------+
only showing top 10 rows


## 7. Model Evaluation & Performance Metrics

In [ ]:
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

accuracy = evaluator_acc.evaluate(predictions)
precision = evaluator_prec.evaluate(predictions)
recall = evaluator_rec.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)

print("\n" + "="*45)
print("     LINEAR SVM CLASSIFICATION METRICS      ")
print("="*45)
print(f" Accuracy  : {accuracy * 100:.2f}%")
print(f" Precision : {precision * 100:.2f}%")
print(f" Recall    : {recall * 100:.2f}%")
print(f" F1-Score  : {f1_score * 100:.2f}%")
print("="*45)


     LINEAR SVM CLASSIFICATION METRICS      
 Accuracy  : 83.59%
 Precision : 83.60%
 Recall    : 83.59%
 F1-Score  : 83.59%
